# HireLens: train, export and evaluate the model

Runs every GPU step of `apps/ml-service/training/README.md` on Google Colab:
**train → evaluate adapter → merge → quantize to GGUF → evaluate the GGUF with the serving runtime.**

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. Prepare the dataset on your own machine first (needs a teacher API key and the local Postgres knowledge base;
   see "Prepare the dataset" in `training/README.md`). You need `data/train.jsonl` and `data/heldout.jsonl`.
3. Put those two files in Google Drive at `MyDrive/hirelens/data/`.

Everything this notebook produces is saved under `MyDrive/hirelens/runs/<timestamp>/`, so a disconnected
session does not lose finished steps. Expect roughly 1–2 hours on a T4 for ~300 examples.

In [ ]:
# Settings
REPO_URL = "https://github.com/Weasley18/hirelens"
REPO_BRANCH = "main"          # change if the training changes are on another branch
EPOCHS = 3
LORA_RANK = 16
QUANT = "Q4_K_M"
# llama.cpp commit vendored by llama-cpp-python==0.3.2 (the serving runtime in requirements.txt).
# Converting with the runtime's own llama.cpp guarantees the GGUF loads in production.
LLAMA_CPP_REVISION = "74d73dc85cc2057446bf63cc37ff649ae7cebd80"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive, clone the repo, bring in the dataset

In [ ]:
import datetime, os, pathlib, shutil
from google.colab import drive
drive.mount("/content/drive")

DRIVE = pathlib.Path("/content/drive/MyDrive/hirelens")
RUN = DRIVE / "runs" / datetime.datetime.now().strftime("%Y%m%d-%H%M")
RUN.mkdir(parents=True, exist_ok=True)

!rm -rf /content/hirelens && git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} /content/hirelens
ML = pathlib.Path("/content/hirelens/apps/ml-service")
(ML / "data").mkdir(exist_ok=True)
for name in ("train.jsonl", "heldout.jsonl"):
    src = DRIVE / "data" / name
    assert src.exists(), f"Missing {src}: prepare the dataset locally and upload it first"
    shutil.copy(src, ML / "data" / name)
os.chdir(ML)
!wc -l data/train.jsonl data/heldout.jsonl
print("Run folder:", RUN)

## 2. Install the training stack and record it

`training.train` refuses to run unless the installed packages exactly match a lock file, so the lock is
captured right after installing. It is saved with the run as provenance.

In [ ]:
!pip install -q unsloth
!pip install -q jsonschema==4.23.0 python-dotenv==1.0.1
!python -m pip freeze > training.lock
shutil.copy("training.lock", RUN / "training.lock")

In [ ]:
# Pin the exact base-model snapshot (an immutable 40-character commit).
from huggingface_hub import HfApi
MODEL_REVISION = HfApi().model_info(BASE_MODEL).sha
print(BASE_MODEL, MODEL_REVISION)
(RUN / "base-model-revision.txt").write_text(f"{BASE_MODEL} {MODEL_REVISION}\n")

## 3. Train the LoRA adapter

In [ ]:
!python -m training.train --model-revision {MODEL_REVISION} --lock-file training.lock --epochs {EPOCHS} --rank {LORA_RANK}
shutil.copytree("hirelens-qwen-lora", RUN / "hirelens-qwen-lora", dirs_exist_ok=True)

## 4. Evaluate the adapter on the held-out set

Fails (non-zero exit) below 95% schema validity or self-consistency. If it fails, fix the dataset before
training longer; more epochs rarely help.

In [ ]:
!python -m training.evaluate
shutil.copy("data/evaluation-report.json", RUN / "evaluation-report.json")

## 5. Merge the adapter into full weights

In [ ]:
!python -m training.merge_and_export

## 6. Convert and quantize to GGUF

Runs in its own Python 3.12 virtual environment: the pinned llama.cpp converter requires
`torch~=2.2.1`, which has no wheels for newer Python versions.

In [ ]:
!pip install -q uv && uv venv --seed --python 3.12 /content/qenv
!PATH=/content/qenv/bin:$PATH LLAMA_CPP_REVISION={LLAMA_CPP_REVISION} QUANT={QUANT} bash scripts/quantize.sh hirelens-merged
for name in ("hirelens-q4.gguf", "hirelens-q4.manifest.json", "quantization-requirements.lock"):
    shutil.copy(name, RUN / name)
!ls -lh hirelens-q4.gguf && sha256sum hirelens-q4.gguf

## 7. Evaluate the GGUF with the serving runtime

Uses `llama-cpp-python==0.3.2`, the same version the inference service ships, with the same context size,
temperature and stop token. The CUDA wheel is used only to make this faster; the production image runs on CPU.
If the CUDA wheel is unavailable, the fallback builds the CPU version (slower: allow 30–60 minutes).

In [ ]:
!python -m venv /content/genv
!/content/genv/bin/pip install -q jsonschema==4.23.0 python-dotenv==1.0.1
rc = os.system("/content/genv/bin/pip install -q llama-cpp-python==0.3.2 --only-binary=:all: "
               "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124")
GPU_LAYERS = -1 if rc == 0 else 0
if rc != 0:
    print("CUDA wheel unavailable; building the CPU runtime")
    !/content/genv/bin/pip install -q llama-cpp-python==0.3.2
!/content/genv/bin/python -m training.evaluate --gguf hirelens-q4.gguf --gpu-layers {GPU_LAYERS}
shutil.copy("data/evaluation-report-gguf.json", RUN / "evaluation-report-gguf.json")

## 8. Done: what to keep

`MyDrive/hirelens/runs/<timestamp>/` now holds:

| File | Use |
|---|---|
| `hirelens-q4.gguf` | The model. Copy to `apps/ml-service/hirelens-q4.gguf` before building the production image. Never commit it. |
| `hirelens-q4.manifest.json` | Provenance: base model + revision, dataset and prompt hashes, llama.cpp commit, GGUF SHA-256 |
| `training.lock`, `quantization-requirements.lock` | Exact package versions used |
| `evaluation-report.json`, `evaluation-report-gguf.json` | Mechanical quality gates for the adapter and the shipped file |

Next: read some of the GGUF outputs yourself, run `python -m eval.run_eval` against a local stack with
`MODEL_BACKEND=gguf`, and measure latency on the target CPU before deploying.

In [ ]:
!ls -lh {RUN}